# PTB-XL Waveform 1D CNN Training & Leak-Free Evaluation

This notebook trains a 1D CNN on 12-lead 100Hz raw ECG signals from PTB-XL using the official patient-stratified split (strat_fold).

### Instructions for Kaggle GPU:
1. Click **Session Options** -> **Accelerator** -> **GPU T4 x2** (or GPU P100).
2. Click **Run All**.
3. The notebook will download PTB-XL, train the 1D CNN model, evaluate on fold 10, and compute 95% bootstrap confidence intervals.


In [ ]:
!pip install -q wfdb pandas numpy scikit-learn tensorflow matplotlib


In [ ]:
import os, sys, json, ast
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix, classification_report
from sklearn.utils.class_weight import compute_class_weight
import wfdb

print("TF Version:", tf.__version__)
print("GPU Available:", bool(tf.config.list_physical_devices("GPU")))


In [ ]:
# Load PTB-XL database metadata and SCP statements
db_url = "https://physionet.org/content/ptb-xl/1.0.3/ptbxl_database.csv"
scp_url = "https://physionet.org/content/ptb-xl/1.0.3/scp_statements.csv"

df_db = pd.read_csv(db_url, index_col="ecg_id")
df_scp = pd.read_csv(scp_url, index_col=0)

df_db["scp_codes"] = df_db["scp_codes"].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)

def assign_label(scp_dict):
    if not isinstance(scp_dict, dict) or len(scp_dict) == 0:
        return None
    valid_classes = set()
    for code, likelihood in scp_dict.items():
        if likelihood >= 50.0 and code in df_scp.index:
            sc = df_scp.loc[code, "diagnostic_class"]
            if pd.notna(sc) and sc != "":
                valid_classes.add(str(sc))
    if not valid_classes:
        return None
    if "MI" in valid_classes:
        return "MI"
    if valid_classes == {"NORM"}:
        return "NORM"
    return "OTHER_ABNORMAL"

df_db["label"] = df_db["scp_codes"].apply(assign_label)
valid_mask = df_db["label"].notna()
print("Total records:", len(df_db), "Usable labeled records:", valid_mask.sum())
clean_df = df_db[valid_mask].copy()


In [ ]:
# Download PTB-XL 100Hz signals via WFDB
os.makedirs("ptbxl_data", exist_ok=True)
print("Downloading 100Hz records...")
wfdb.dl_database("ptb-xl", dl_dir="ptbxl_data", records="records100")


In [ ]:
# Load signals into numpy arrays
labels = ["MI", "NORM", "OTHER_ABNORMAL"]
label_to_id = {l: i for i, l in enumerate(labels)}

X_list, y_list, folds_list = [], [], []
for ecg_id, row in clean_df.iterrows():
    rec_path = os.path.join("ptbxl_data", row["filename_lr"])
    try:
        record = wfdb.rdrecord(rec_path)
        signal = record.p_signal
        if signal.shape == (1000, 12):
            X_list.append(signal)
            y_list.append(label_to_id[row["label"]])
            folds_list.append(row["strat_fold"])
    except Exception as e:
        pass

X = np.array(X_list, dtype=np.float32)
y = np.array(y_list, dtype=np.int32)
folds = np.array(folds_list, dtype=np.int32)
print("Loaded X shape:", X.shape, "y shape:", y.shape)


In [ ]:
# Official Split (1-8 Train, 9 Val, 10 Test)
train_mask = np.isin(folds, range(1, 9))
val_mask = (folds == 9)
test_mask = (folds == 10)

X_train, y_train = X[train_mask], y[train_mask]
X_val, y_val = X[val_mask], y[val_mask]
X_test, y_test = X[test_mask], y[test_mask]

cw_vec = compute_class_weight(class_weight="balanced", classes=np.array([0, 1, 2]), y=y_train)
class_weights = dict(zip([0, 1, 2], cw_vec))
print("Class weights:", class_weights)


In [ ]:
# Build 1D CNN Architecture
def build_1d_cnn(input_shape=(1000, 12), num_classes=3):
    model = models.Sequential([
        layers.Conv1D(32, kernel_size=7, padding="same", activation="relu", input_shape=input_shape),
        layers.BatchNormalization(),
        layers.MaxPooling1D(2),
        layers.Conv1D(64, kernel_size=5, padding="same", activation="relu"),
        layers.BatchNormalization(),
        layers.MaxPooling1D(2),
        layers.Conv1D(128, kernel_size=3, padding="same", activation="relu"),
        layers.BatchNormalization(),
        layers.GlobalAveragePooling1D(),
        layers.Dense(64, activation="relu"),
        layers.Dropout(0.3),
        layers.Dense(num_classes, activation="softmax")
    ])
    model.compile(
        optimizer=optimizers.Adam(learning_rate=1e-3),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model

model = build_1d_cnn()
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=20,
    batch_size=64,
    class_weight=class_weights,
    verbose=1
)


In [ ]:
# Evaluate ONCE on Test Fold (Fold 10)
test_preds_prob = model.predict(X_test)
test_preds = np.argmax(test_preds_prob, axis=1)

test_acc = accuracy_score(y_test, test_preds)
test_macro_f1 = f1_score(y_test, test_preds, average="macro")
y_test_cat = tf.keras.utils.to_categorical(y_test, 3)
test_auroc = roc_auc_score(y_test_cat, test_preds_prob, multi_class="ovr", average="macro")

print("Test Accuracy:", test_acc)
print("Test Macro F1:", test_macro_f1)
print("Test Macro AUROC:", test_auroc)
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, test_preds))
print("\nClassification Report:")
print(classification_report(y_test, test_preds, target_names=labels))

# Bootstrap 95% CIs
np.random.seed(42)
n_bootstraps = 1000
boot_f1s, boot_accs, boot_aurocs = [], [], []
n_samples = len(y_test)
for _ in range(n_bootstraps):
    indices = np.random.choice(n_samples, size=n_samples, replace=True)
    if len(np.unique(y_test[indices])) < 3:
        continue
    boot_accs.append(accuracy_score(y_test[indices], test_preds[indices]))
    boot_f1s.append(f1_score(y_test[indices], test_preds[indices], average="macro"))
    try:
        auroc = roc_auc_score(y_test_cat[indices], test_preds_prob[indices], multi_class="ovr", average="macro")
        boot_aurocs.append(auroc)
    except:
        pass

f1_ci = np.percentile(boot_f1s, [2.5, 97.5])
acc_ci = np.percentile(boot_accs, [2.5, 97.5])
auroc_ci = np.percentile(boot_aurocs, [2.5, 97.5])

print("Macro F1 95% CI:", f1_ci)
print("Accuracy 95% CI:", acc_ci)
print("Macro AUROC 95% CI:", auroc_ci)
